## Loading the IMDB dataset

In [2]:
vocab_size=10000

from tensorflow.keras.datasets import imdb

(train_data, train_labels), (test_data, test_labels) = imdb.load_data(num_words=vocab_size)

17464789/17464789 [==============================] - 0s 0us/step


In [2]:
for doc_index in range(0,10):
  print(f'[{doc_index}] \t label:{train_labels[doc_index]} \tdoc length:{str(len(train_data[doc_index])).ljust(4)}   doc[0:20]:{train_data[doc_index][0:20]}')

[0] 	 label:1 	doc length:218    doc[0:20]:[1, 14, 22, 16, 43, 530, 973, 1622, 1385, 65, 458, 4468, 66, 3941, 4, 173, 36, 256, 5, 25]
[1] 	 label:0 	doc length:189    doc[0:20]:[1, 194, 1153, 194, 8255, 78, 228, 5, 6, 1463, 4369, 5012, 134, 26, 4, 715, 8, 118, 1634, 14]
[2] 	 label:0 	doc length:141    doc[0:20]:[1, 14, 47, 8, 30, 31, 7, 4, 249, 108, 7, 4, 5974, 54, 61, 369, 13, 71, 149, 14]
[3] 	 label:1 	doc length:550    doc[0:20]:[1, 4, 2, 2, 33, 2804, 4, 2040, 432, 111, 153, 103, 4, 1494, 13, 70, 131, 67, 11, 61]
[4] 	 label:0 	doc length:147    doc[0:20]:[1, 249, 1323, 7, 61, 113, 10, 10, 13, 1637, 14, 20, 56, 33, 2401, 18, 457, 88, 13, 2626]
[5] 	 label:0 	doc length:43     doc[0:20]:[1, 778, 128, 74, 12, 630, 163, 15, 4, 1766, 7982, 1051, 2, 32, 85, 156, 45, 40, 148, 139]
[6] 	 label:1 	doc length:123    doc[0:20]:[1, 6740, 365, 1234, 5, 1156, 354, 11, 14, 5327, 6638, 7, 1016, 10626, 5940, 356, 44, 4, 1349, 500]
[7] 	 label:0 	doc length:562    doc[0:20]:[1, 4, 2, 716, 4, 65, 7

In [2]:
max([max(sequence) for sequence in train_data])

9999

## Decoding reviews back to text

In [3]:
word_index = imdb.get_word_index()
reverse_word_index = dict(
    [(value, key) for (key, value) in word_index.items()])


1641221/1641221 [==============================] - 0s 0us/step


In [4]:
input_word = "amazing"
index = word_index.get(input_word)

if index is not None:
    print(f"The index of '{input_word}' is: {index}")
else:
    print(f"'{input_word}' is not found in the IMDb dataset.")

The index of 'amazing' is: 477


In [14]:
for doc_index in range(0,10):
  decoded_review = " ".join([reverse_word_index.get(i - 3, "?") for i in train_data[doc_index]])
  print(f'[{doc_index}]: {decoded_review}')

[0]: ? this film was just brilliant casting location scenery story direction everyone's really suited the part they played and you could just imagine being there robert ? is an amazing actor and now the same being director ? father came from the same scottish island as myself so i loved the fact there was a real connection with this film the witty remarks throughout the film were great it was just brilliant so much that i bought the film as soon as it was released for ? and would recommend it to everyone to watch and the fly fishing was amazing really cried at the end it was so sad and you know what they say if you cry at a film it must have been good and this definitely was also congratulations to the two little boy's that played the ? of norman and paul they were just brilliant children are often left out of the praising list i think because the stars that play them all grown up are such a big profile for the whole film but these children are amazing and should be praised for what th

## Encoding the integer sequences via multi-hot encoding

In [4]:
import numpy as np
def vectorize_sequences(sequences, dimension=vocab_size):
    results = np.zeros((len(sequences), dimension))
    for i, sequence in enumerate(sequences):
        for j in sequence:
            results[i, j] = 1.
    return results
x_train = vectorize_sequences(train_data)
x_test = vectorize_sequences(test_data)

In [15]:
len(train_data[0])

218

In [16]:
len(x_train[0])

18000

In [10]:
x_train.shape

(25000, 18000)

In [17]:
train_data.shape

(25000,)

In [5]:
y_train = np.asarray(train_labels).astype("float32")
y_test = np.asarray(test_labels).astype("float32")

## Building your model

In [11]:
from tensorflow import keras
from keras import layers
from keras import activations

model = keras.Sequential([
    layers.Dense(10, activation="relu", input_shape=(vocab_size,)),
    layers.Dense(1024, activation="relu"),  #First Hidden Layer
    layers.Dense(1024, activation="relu"),  #Second Hidden Layer

    layers.Dense(1, activation="sigmoid")
])

In [12]:
model.compile(optimizer="rmsprop",
              loss="binary_crossentropy",
              metrics=["accuracy"])

In [13]:
model.build()

In [14]:
model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_4 (Dense)             (None, 10)                100010    
                                                                 
 dense_5 (Dense)             (None, 1024)              11264     
                                                                 
 dense_6 (Dense)             (None, 1024)              1049600   
                                                                 
 dense_7 (Dense)             (None, 1)                 1025      
                                                                 
Total params: 1,161,899
Trainable params: 1,161,899
Non-trainable params: 0
_________________________________________________________________


In [15]:
model.fit(x_train, y_train, epochs=4, batch_size=512)

Epoch 1/4
49/49 [==============================] - 6s 105ms/step - loss: 0.4644 - accuracy: 0.7706
Epoch 2/4
49/49 [==============================] - 4s 90ms/step - loss: 0.2556 - accuracy: 0.8997
Epoch 3/4
49/49 [==============================] - 5s 109ms/step - loss: 0.1986 - accuracy: 0.9256
Epoch 4/4
49/49 [==============================] - 4s 90ms/step - loss: 0.1576 - accuracy: 0.9393


In [16]:
test_loss, test_acc = model.evaluate(x_test, y_test)
print(f"test_acc: {test_acc} \t test_loss: {test_loss}")

782/782 [==============================] - 4s 5ms/step - loss: 0.3184 - accuracy: 0.8766
test_acc: 0.8766000270843506 	 test_loss: 0.3183525502681732
